In [1]:
import pandas as pd
import numpy as np

from src.connection import local_client as client

In [2]:
words = [
    "cancer",
    "neuroscience",
    "cardiology",
    "ecology",
    "bioinformatics",
    "chemistry",
    "surgery",
    "environment",
    "material",
    "microbiology",
    "pediatric",
    "immunology",
    "psychology",
    "psychiatry",
    "genetics",
    "nutrition",
    "veterinary",
    "engineering",
    "education",
    "physics",
    "optics",
    "nursing",
    "neurology",
    "radiology",
    "ophthalmology",
    "gynecology",
    "rehabilitation",
    "pathology",
    "anesthesiology",
    "dermatology",
    "pharmacology",
    "physiology",
    "virology",
    "biochemistry",
    "computation",
    "infectious",
    "healthcare",
    "ethics",
]

In [3]:
# Index
index_name_pubmed = "frameintell_pubmed"
index_name_insights = "frameintell_insights"

# Define the dim_reduction_method and clustering_method criteria
reference_text = "abstract"
dim_reduction_method = "tsne"
clustering_method = "kmeans"

In [4]:
query_journal = {"_source": ["journalInformation.journalTitle"], "query": {"match_all": {}}}

query_insights = {"_source": ["_id","insights.clustering_information.label"],
                  "query": {"bool": {"must": [
                           {"term": {"reference_text":reference_text}},
                           {"nested": {
                               "path": "insights",
                               "query": {"term": {"insights.dim_reduction_method": dim_reduction_method}}
                            }},
                           {"nested": {
                               "path": "insights.clustering_information",
                               "query": {"term": {"insights.clustering_information.clustering_method": clustering_method}}
                            }}
                    ]}}}

In [5]:
def fetch_data(client,query,index):
    data = client.search(
            body = query,
            index = index,
            size=10000,
            scroll = '10m'
        )

    scroll_id = data['_scroll_id']
    scroll_size = len(data['hits']['hits'])

    output = data['hits']['hits']
    while scroll_size>0:
        print(scroll_size,scroll_id)
        data = client.scroll(
                scroll_id=scroll_id, 
                scroll='10m'
            )
        scroll_id = data['_scroll_id']
        scroll_size = len(data['hits']['hits'])

        output = output + data['hits']['hits']

    return output

In [6]:
journal_data = fetch_data(client,query_journal,index_name_pubmed)

journals = []
for item in journal_data:
    journals.append({"_id":item["_id"],
                     "journal_title":item["_source"]["journalInformation"]["journalTitle"]})
journals_df = pd.DataFrame(journals)
print(journals_df.head(3))

10000 FGluY2x1ZGVfY29udGV4dF91dWlkDXF1ZXJ5QW5kRmV0Y2gBFjNWZVFGd1hCVHVpRzFBRXI2U3BPU3cAAAAAAAAFrxZMMWVLRTJfblF4eXl4TzVoUnJvRUhn
10000 FGluY2x1ZGVfY29udGV4dF91dWlkDXF1ZXJ5QW5kRmV0Y2gBFjNWZVFGd1hCVHVpRzFBRXI2U3BPU3cAAAAAAAAFrxZMMWVLRTJfblF4eXl4TzVoUnJvRUhn
1769 FGluY2x1ZGVfY29udGV4dF91dWlkDXF1ZXJ5QW5kRmV0Y2gBFjNWZVFGd1hCVHVpRzFBRXI2U3BPU3cAAAAAAAAFrxZMMWVLRTJfblF4eXl4TzVoUnJvRUhn
        _id                                      journal_title
0  31655504                Turkish journal of medical sciences
1  31586390                 The Journal of infectious diseases
2  31287549  Nicotine & tobacco research : official journal...


In [7]:
insights_data = fetch_data(client,query_insights,index_name_insights)

insights = []
for item in insights_data:
    insights.append({"_id":item["_id"].split(":")[1],
                     "label":item["_source"]["insights"][0]["clustering_information"][0]["label"]})
insights_df = pd.DataFrame(insights)
print(insights_df.head(3))

10000 FGluY2x1ZGVfY29udGV4dF91dWlkDXF1ZXJ5QW5kRmV0Y2gBFjNWZVFGd1hCVHVpRzFBRXI2U3BPU3cAAAAAAAAFsBZMMWVLRTJfblF4eXl4TzVoUnJvRUhn
10000 FGluY2x1ZGVfY29udGV4dF91dWlkDXF1ZXJ5QW5kRmV0Y2gBFjNWZVFGd1hCVHVpRzFBRXI2U3BPU3cAAAAAAAAFsBZMMWVLRTJfblF4eXl4TzVoUnJvRUhn
1769 FGluY2x1ZGVfY29udGV4dF91dWlkDXF1ZXJ5QW5kRmV0Y2gBFjNWZVFGd1hCVHVpRzFBRXI2U3BPU3cAAAAAAAAFsBZMMWVLRTJfblF4eXl4TzVoUnJvRUhn
        _id  label
0  30597315     31
1  30596972      4
2  30596169      9


In [8]:
journal_label_df = pd.merge(journals_df, insights_df, on='_id', how='inner')
print(journal_label_df.head(3))

        _id                                      journal_title  label
0  31655504                Turkish journal of medical sciences     20
1  31586390                 The Journal of infectious diseases      6
2  31287549  Nicotine & tobacco research : official journal...      7


In [9]:
journals = journal_label_df["journal_title"]
    
labels=np.zeros(len(journal_label_df))

for i, wrd in enumerate(words):

    word_may = wrd.capitalize()
    word_min = ' '+wrd

    indexes1 = journals.str.find(word_may) 
    indexes2 = journals.str.find(word_min)

    labels = np.where((indexes1!=-1) | (indexes2!=-1), wrd, labels)

In [10]:
labeled = np.where(labels=="0.0", False, True)
print(labeled.sum()/len(labeled)*100,"%")

33.648766594698884 %


In [11]:
journal_label_df['category'] = labels
print(journal_label_df[["_id","label","category"]].head(3))

        _id  label    category
0  31655504     20         0.0
1  31586390      6  infectious
2  31287549      7         0.0


In [14]:
df_filtered = journal_label_df[journal_label_df['category'] != '0.0']
label_category = []
for label,group in df_filtered.groupby('label'):
    print(label,group['category'].value_counts(normalize=True)[:3],"\n")
    category = "_".join(group['category'].value_counts(normalize=True)[:3].index.tolist())    
    label_category.append({'label':label,'category':category})

0 chemistry    0.556180
material     0.213483
physics      0.095506
Name: category, dtype: float64 

1 environment    0.213592
neurology      0.145631
psychiatry     0.106796
Name: category, dtype: float64 

2 surgery      0.488636
material     0.125000
pediatric    0.096591
Name: category, dtype: float64 

3 environment    0.891738
material       0.028490
ecology        0.022792
Name: category, dtype: float64 

4 cancer        0.488987
immunology    0.105727
pathology     0.092511
Name: category, dtype: float64 

5 rehabilitation    0.350515
neurology         0.103093
engineering       0.082474
Name: category, dtype: float64 

6 microbiology    0.300000
infectious      0.254545
immunology      0.095455
Name: category, dtype: float64 

7 psychology    0.320513
psychiatry    0.307692
nursing       0.098291
Name: category, dtype: float64 

8 material     0.377451
chemistry    0.299020
physics      0.151961
Name: category, dtype: float64 

9 nutrition     0.255952
cardiology    0.172619
p

In [17]:
import json

with open('../website/api-fast/configure/topic_labels.json','w') as f:
    json.dump(label_category, f)